# provrep on Google Colab — full-graph GPU training

Provenance-Grounded Drug Repurposing. Team 14, KMIT CSD A.

**First:** Runtime menu -> Change runtime type -> T4 GPU -> Save.

Then run the cells top to bottom (Shift+Enter). You upload one file, provrep.zip (~0.1 MB).
Everything else, including the Hetionet data, downloads inside Colab — nothing large touches your PC.
About 30-60 minutes total, mostly training. Keep this tab open.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then re-run."

## 2. Upload provrep.zip
A file picker appears. Choose **provrep.zip**.

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
zip_name = next(n for n in uploaded if n.endswith(".zip"))
zipfile.ZipFile(zip_name).extractall("/content")
%cd /content/provrep
print(sorted(os.listdir(".")))

## 3. Install the project
PyTorch is already in Colab, so this only adds the smaller libraries (~1 min).

In [ ]:
!pip install -q -e ".[dev]"
!python -m pytest -q -p no:cacheprovider 2>&1 | tail -2

## 4. Colab GPU config
Full graph (all 2.25M edges), dimension 128. A T4 has 16 GB, so this fits easily.

In [ ]:
%%writefile configs/colab.yaml
inherit: default.yaml

paths:
  work_dir: artifacts/gpu

data:
  drop_metaedges: []        # keep every edge type (the CPU run dropped some to save time)

scorer:
  hidden_dim: 128
  num_bases: 16
  epochs: 800
  patience: 150
  device: cuda

## 5. Download Hetionet and build the knowledge graph
About 15 MB, downloaded inside Colab.

In [ ]:
!python -m provrep.cli -c configs/colab.yaml download
!python -m provrep.cli -c configs/colab.yaml build-kg 2>&1 | tail -25

## 6. Train the R-GCN on the GPU
The long step (~20-45 min). A progress line prints every 10 epochs.

In [ ]:
!python -m provrep.cli -c configs/colab.yaml train 2>&1 | grep -E "epoch|early stopping|map|recall|weights|Error|Traceback"

## 7. Calibrate and evaluate on the held-out test diseases

In [ ]:
!python -m provrep.cli -c configs/colab.yaml calibrate 2>&1 | grep -E "calibrators fitted|Error|Traceback"
!python -m provrep.cli -c configs/colab.yaml eval > /dev/null 2>&1
from IPython.display import Markdown, Image, display
display(Markdown(open("artifacts/gpu/eval/eval_report.md").read()))
display(Image("artifacts/gpu/eval/reliability.png", width=480))

## 8. Demo: triage diseases the model never saw a treatment for

In [ ]:
for disease in ["rheumatoid arthritis", "migraine"]:
    !python -m provrep.cli -c configs/colab.yaml triage "{disease}" --show 10 2>&1 | grep -v -E "INFO|WARNING"

## 9. Download the results
Saves the trained model and report as a zip, then downloads it to your computer.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/provrep_colab_results", "zip", "artifacts/gpu")
files.download("/content/provrep_colab_results.zip")

---
### Optional: real Claude agents instead of the offline stand-in

Needs your own Anthropic API key. Paste it below, then re-run a triage.
Each triage makes roughly 60 model calls, so it uses some API credit.

In [ ]:
import getpass, os
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
!pip install -q anthropic
!python -m provrep.cli -c configs/colab.yaml --set agents.llm.provider=anthropic triage "migraine" --show 10 2>&1 | grep -v -E "INFO|WARNING"